In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features.feature_engineer.apm_features import _detect_star_players


In [2]:
def _opponent_season_stats_and_rank(df: pd.DataFrame) -> pd.DataFrame:
    """
    Opponent (team) features: YTD means through prior games only, plus weekly league
    rank among teams (1 = best) using each team's S2D stat at the last game in the
    ISO week, broadcast to all games in that week.
    """
    if 'OPP_OPP_ABBREVIATION_base' not in df.columns or 'SEASON_YEAR' not in df.columns:
        return df

    stat_cols = ['TEAM_DEF_RATING', 'TEAM_PACE', 'TEAM_POSS']
    available = [c for c in stat_cols if c in df.columns]
    if not available:
        return df

    need = ['GAME_ID', 'GAME_DATE', 'SEASON_YEAR', 'TEAM_ABBREVIATION'] + available
    team_game = (
        df[need]
        .drop_duplicates(subset=['GAME_ID', 'TEAM_ABBREVIATION'])
        .copy()
    )
    team_game['GAME_DATE'] = pd.to_datetime(team_game['GAME_DATE'], utc=False)
    team_game = team_game.sort_values(['SEASON_YEAR', 'TEAM_ABBREVIATION', 'GAME_DATE'])

    g = team_game.groupby(['SEASON_YEAR', 'TEAM_ABBREVIATION'], sort=False)

    season_renames = {}
    for col in available:
        base = col.replace('TEAM_', '')
        sname = f'{base}_season_to_date'
        team_game[sname] = g[col].transform(
            lambda x: x.expanding().mean().round(2)
        )
        season_renames[col] = sname

    team_game['RANK_WEEK'] = team_game['GAME_DATE'].dt.to_period('W-SAT')
    last_idx = team_game.groupby(
        ['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION'], sort=False
    )['GAME_DATE'].idxmax()
    weekly = team_game.loc[last_idx].copy()

    # Def rating: lower is better. Pace / poss: higher = rank 1.
    rank_asc = {
        'TEAM_DEF_RATING': True,
        'TEAM_PACE': False,
        'TEAM_POSS': False,
    }
    rank_cols = []
    for col, sname in season_renames.items():
        asc = rank_asc.get(col, True)
        rk = sname.replace('_season_to_date', '_szn_league_rank')
        rank_cols.append(rk)
        weekly[rk] = (
            weekly.groupby(['SEASON_YEAR', 'RANK_WEEK'])[sname]
            .rank(ascending=asc, method='min')
        )

    w = weekly[['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION', *rank_cols]]
    team_game = team_game.merge(
        w, on=['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION'], how='left'
    )

    out_cols = list(season_renames.values()) + rank_cols
    opp_rename = {c: f'OPP_{c}' for c in out_cols}
    team_game_opp = (
        team_game[['GAME_ID', 'TEAM_ABBREVIATION', *out_cols]]
        .rename(columns={**{'TEAM_ABBREVIATION': 'OPP_OPP_ABBREVIATION_base'}, **opp_rename})
    )

    return df.merge(team_game_opp, on=['GAME_ID', 'OPP_OPP_ABBREVIATION_base'], how='left')

In [4]:
# pd.set_option('display.max_columns', None)

# s21 = pd.read_csv('data/raw/season_stats/S21.csv').sort_values(by='GAME_DATE')
# p21 = pd.read_csv('data/raw/playoff_stats/P21.csv').sort_values(by='GAME_DATE')
# s21 = pd.concat([s21, p21])

# s22 = pd.read_csv('data/raw/season_stats/S22.csv').sort_values(by='GAME_DATE')
# p22 = pd.read_csv('data/raw/playoff_stats/P22.csv').sort_values(by='GAME_DATE')
# s22 = pd.concat([s22, p22])

# s23 = pd.read_csv('data/raw/season_stats/S23.csv').sort_values(by='GAME_DATE')
# p23 = pd.read_csv('data/raw/playoff_stats/P23.csv').sort_values(by='GAME_DATE')
# s23 = pd.concat([s23, p23])

s24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S24.csv').sort_values(by='GAME_DATE')
s24 = _detect_star_players(s24)
s24 = _opponent_season_stats_and_rank(s24)
p24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P24.csv').sort_values(by='GAME_DATE')
p24 = _detect_star_players(p24)
p24 = _opponent_season_stats_and_rank(p24)
s24 = pd.concat([s24, p24])

s25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s25 = _detect_star_players(s25)
s25 = _opponent_season_stats_and_rank(s25)
p25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
p25 = _detect_star_players(p25)
p25 = _opponent_season_stats_and_rank(p25)
s25 = pd.concat([s25, p25])

s26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
s26 = _detect_star_players(s26)
s26 = _opponent_season_stats_and_rank(s26)
p26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
p26 = _detect_star_players(p26)
p26 = _opponent_season_stats_and_rank(p26)
s26 = pd.concat([s26, p26])

In [5]:
df = pd.concat([s25,s26])
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df['DAYS_REST'] = (
    df.groupby('PLAYER_ID')['GAME_DATE'].diff().dt.days.fillna(3))
df['IS_B2B']  = (df['DAYS_REST'] == 1).astype(int)
df['IS_HOME'] = df['MATCHUP'].str.contains('vs', na=False).astype(int)
df['TEAM_RATING_SMOOTH'] = df.groupby('TEAM_ID')['TEAM_NET_RATING'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean())
df['OPP_RATING_SMOOTH'] = df.groupby('OPP_TEAM_ID')['OPP_NET_RATING'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean())
df['EXPECTED_SPREAD'] = np.where(
    df['IS_HOME'] == 1,
    ((df['TEAM_RATING_SMOOTH'] - df['OPP_RATING_SMOOTH']) * -1.0) - 3.0,
    ((df['TEAM_RATING_SMOOTH'] - df['OPP_RATING_SMOOTH']) * -1.0) + 3.0)
df['EXPECTED_SPREAD'] = df['EXPECTED_SPREAD'].clip(lower=-20, upper=20)
df['BLOWOUT_RISK'] = (df['EXPECTED_SPREAD'].abs() >= 10).astype(int)
df.sample(10)

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,...,OPP_PACE_szn_league_rank,OPP_POSS_szn_league_rank,name,DAYS_REST,IS_B2B,IS_HOME,TEAM_RATING_SMOOTH,OPP_RATING_SMOOTH,EXPECTED_SPREAD,BLOWOUT_RISK
25681,624,2024-25,1631311,Lester Quinones,Lester,1610612740,NOP,New Orleans Pelicans,22401161,2025-04-10,...,14.0,14.0,NaN,2.0,0,0,-18.18,18.60,20.00,1
17817,8828,2025-26,1629614,Andrew Nembhard,Andrew,1610612754,IND,Indiana Pacers,22500795,2026-02-19,...,6.0,6.0,NaN,9.0,0,0,1.10,-11.72,-9.82,0
17023,9623,2025-26,201587,Nicolas Batum,Nicolas,1610612746,LAC,LA Clippers,22500760,2026-02-08,...,9.0,8.0,NaN,2.0,0,0,8.66,-10.70,-16.36,1
4177,22473,2025-26,1630579,Jericho Sims,Jericho,1610612749,MIL,Milwaukee Bucks,22500231,2025-11-15,...,20.0,24.0,NaN,1.0,1,1,-5.75,21.10,20.00,1
14213,12073,2024-25,1631209,Isaiah Wong,Isaiah,1610612766,CHA,Charlotte Hornets,22400637,2025-01-25,...,15.0,16.0,NaN,1.0,1,1,-1.49,-16.15,-17.66,1
13856,12443,2024-25,1630163,LaMelo Ball,LaMelo,1610612766,CHA,Charlotte Hornets,22400616,2025-01-22,...,1.0,2.0,NaN,2.0,0,0,-11.50,11.50,20.00,1
18575,7734,2024-25,1641796,Pelle Larsson,Pelle,1610612748,MIA,Miami Heat,22400841,2025-02-26,...,2.0,1.0,NaN,13.0,0,1,-6.04,6.04,9.08,0
14645,12003,2025-26,1630596,Evan Mobley,Evan,1610612739,CLE,Cleveland Cavaliers,22500646,2026-01-24,...,17.0,15.0,NaN,1.0,1,0,11.81,-19.16,-20.00,1
20238,6408,2025-26,203114,Khris Middleton,Khris,1610612742,DAL,Dallas Mavericks,22500899,2026-03-05,...,18.0,15.0,NaN,2.0,0,0,-13.20,8.12,20.00,1
21654,4981,2025-26,201572,Brook Lopez,Brook,1610612746,LAC,LA Clippers,22500975,2026-03-14,...,15.0,18.0,NaN,1.0,1,1,6.64,-2.72,-12.36,1


In [ ]:
# adjustments = {
#     # Still manual — model can't learn these from IS_PLAYOFF alone
#     'same_day_injury':      '±X',   # lineup news after model runs
#     'star_out_context':     '±X',   # who specifically is out matters
#     'series_specific':      '±X',   # playoff series fatigue, game 1 vs game 7
#     'blowout_risk':         '±X',   # spread context
#     'minutes_restriction':  '±X',   # coach explicitly limiting player
# }

# adjustments = {
#     'away_game':        -1.18,  # still valid, model has IS_HOME
#     'hot_streak':       +0.50,  # last 3 at 17.33, recency not in model
#     'series_context':    0.00,  # game 1-3, no fatigue yet
#     'blowout_risk':      0.00,  # flat for Scoot
# }

In [6]:
player_name = 'Scoot Henderson'
stat_name = 'PTS'

pdf = df[df['PLAYER_NAME'] == player_name].sort_values(by='GAME_DATE')
okc_stats = pdf[pdf['OPP_TEAM_ID'] == 1610612760][stat_name].mean()
home_stats = pdf[pdf['IS_HOME'] == 1][stat_name]
away_stats = pdf[pdf['IS_HOME'] == 0][stat_name]
b2b_stats = pdf[pdf['IS_B2B'] == 1][stat_name]
not_b2b_stats = pdf[pdf['IS_B2B'] == 0]
res_days_2_stats = pdf[pdf['DAYS_REST'] == 2]
res_days_3_stats = pdf[pdf['DAYS_REST'] == 3]
active_star_count_1 = pdf[pdf['ACTIVE_STARS_COUNT'] == 1]
active_star_count_2 = pdf[pdf['ACTIVE_STARS_COUNT'] == 2]
active_star_count_3 = pdf[pdf['ACTIVE_STARS_COUNT'] == 3]
star_out_flag_1 = pdf[pdf['TOP_STAR_ACTIVE'] == 1]
star_out_flag_0 = pdf[pdf['TOP_STAR_ACTIVE'] == 0]
first_20_games = pdf[stat_name].iloc[:20].mean()
last_20_games = pdf[stat_name].iloc[-20:].mean()
last_3_games = pdf[stat_name].iloc[-3:].mean()
# blowout_risk_1 = pdf[pdf['BLOWOUT_RISK'] == 1]
# blowout_risk_0 = pdf[pdf['BLOWOUT_RISK'] == 0]
favorite_team = pdf[pdf['EXPECTED_SPREAD'] < 0]
heavy_favorite_team = pdf[pdf['EXPECTED_SPREAD'] < -10]
underdog_team = pdf[pdf['EXPECTED_SPREAD'] > 0]
heavy_underdog_team = pdf[pdf['EXPECTED_SPREAD'] > 10]
top_opponent_def_rating = pdf[pdf['OPP_DEF_RATING_szn_league_rank'] <= 5]
middle_opponent_def_rating = pdf[(pdf['OPP_DEF_RATING_szn_league_rank'] > 5) & (pdf['OPP_DEF_RATING_szn_league_rank'] <= 15)]
bottom_opponent_def_rating = pdf[pdf['OPP_DEF_RATING_szn_league_rank'] > 15]
high_opponent_pace = pdf[pdf['OPP_PACE_szn_league_rank'] <= 5]
middle_opponent_pace = pdf[(pdf['OPP_PACE_szn_league_rank'] > 5) & (pdf['OPP_PACE_szn_league_rank'] <= 15)]
low_opponent_pace = pdf[pdf['OPP_PACE_szn_league_rank'] > 15]
playoff_game = pdf[pdf['IS_PLAYOFF'] == 1]
not_playoff_game = pdf[pdf['IS_PLAYOFF'] == 0]


print(f"{pdf['PLAYER_NAME'].iloc[0]} | AVG {stat_name}: {pdf[stat_name].mean().round(2)}")
print(f"First 20 games: {first_20_games.round(2)}")
print(f"Last 20 games: {last_20_games.round(2)}")
print(f"Last 3 games: {last_3_games.round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Home: {home_stats.mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Away: {away_stats.mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} B2B: {b2b_stats.mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Not B2B: {not_b2b_stats[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Rest 2 Days: {res_days_2_stats[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Rest 3 Days: {res_days_3_stats[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 1: {round(active_star_count_1[stat_name].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 2: {round(active_star_count_2[stat_name].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 3: {round(active_star_count_3[stat_name].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Favorite Team: {favorite_team[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Heavy Favorite Team: {heavy_favorite_team[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Underdog Team: {underdog_team[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Heavy Underdog Team: {heavy_underdog_team[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Top Opponent Def Rating: {top_opponent_def_rating[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Middle Opponent Def Rating: {middle_opponent_def_rating[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Bottom Opponent Def Rating: {bottom_opponent_def_rating[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} High Opponent Pace: {high_opponent_pace[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Middle Opponent Pace: {middle_opponent_pace[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Low Opponent Pace: {low_opponent_pace[stat_name].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Playoff Game {stat_name} avg: {round(playoff_game[stat_name].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Not Playoff Game {stat_name} avg: {round(not_playoff_game[stat_name].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Playoff Game MIN avg: {round(playoff_game['MIN'].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Not Playoff Game MIN avg: {round(not_playoff_game['MIN'].mean(), 2)}")



Scoot Henderson | AVG PTS: 13.26
First 20 games: 10.65
Last 20 games: 14.6
Last 3 games: 8.67
Scoot Henderson Home: 14.47
Scoot Henderson Away: 12.02
Scoot Henderson B2B: 13.4
Scoot Henderson Not B2B: 13.23
Scoot Henderson Rest 2 Days: 12.91
Scoot Henderson Rest 3 Days: 17.7
Scoot Henderson Active Star Count 1: 13.47
Scoot Henderson Active Star Count 2: 15.14
Scoot Henderson Active Star Count 3: 11.49
Scoot Henderson Favorite Team: 13.51
Scoot Henderson Heavy Favorite Team: 13.56
Scoot Henderson Underdog Team: 13.04
Scoot Henderson Heavy Underdog Team: 13.05
Scoot Henderson Top Opponent Def Rating: 12.67
Scoot Henderson Middle Opponent Def Rating: 14.12
Scoot Henderson Bottom Opponent Def Rating: 12.9
Scoot Henderson High Opponent Pace: 12.73
Scoot Henderson Middle Opponent Pace: 14.83
Scoot Henderson Low Opponent Pace: 12.28
Scoot Henderson Playoff Game PTS avg: 15.0
Scoot Henderson Not Playoff Game PTS avg: 13.17
Scoot Henderson Playoff Game MIN avg: 29.02
Scoot Henderson Not Playoff

## HOME vs AWAY player avgs

In [43]:
# Mean minutes per player over all games in s26
min_by_player = df.groupby("PLAYER_ID")["MIN"].mean()
games_by_player = df.groupby("PLAYER_ID")["GAME_ID"].nunique()

eligible_ids = min_by_player.index[
    (min_by_player > 15) & (games_by_player.reindex(min_by_player.index) >= 20)
]

df = df[df["PLAYER_ID"].isin(eligible_ids)].copy()
df["IS_HOME"] = df["MATCHUP"].str.contains("vs", na=False)
pt_home_away = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_HOME"], as_index=False)
    .agg(PTS_PER_GAME=("PTS", "mean"), GAMES=("GAME_ID", "count"))
)

# Wide form: one row per player, columns for home / away
wide = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_HOME"])["PTS"]
    .mean()
    .unstack()
    .rename(columns={False: "away_pts", True: "home_pts"})
)
wide["home_minus_away"] = wide["home_pts"] - wide["away_pts"]
wide = wide.sort_values("home_minus_away", ascending=False)
wide.sample(10)

,IS_HOME,away_pts,home_pts,home_minus_away
PLAYER_ID,PLAYER_NAME,,,
1630536,Sharife Cooper,8.285714,7.950000,-0.335714
1628971,Bruce Brown,9.561798,8.554455,-1.007342
1628365,Markelle Fultz,5.676471,5.914286,0.237815
1626167,Myles Turner,14.454545,15.354545,0.900000
1631200,Kris Murray,4.631579,5.956989,1.325410
1630610,DeJon Jarreau,5.000000,7.818182,2.818182
203081,Damian Lillard,23.606557,25.457143,1.850585
1629216,Gabe Vincent,5.402985,5.260870,-0.142116
203915,Spencer Dinwiddie,11.105263,10.417722,-0.687542


## How players perform on Back 2 Back games

In [46]:
# After filtering to eligible_ids
df = df[df["PLAYER_ID"].isin(eligible_ids)].copy()

# Optional: if IS_B2B is int, this keeps 0/1 unstack labels predictable
# df["IS_B2B"] = df["IS_B2B"].astype(int)

b2b_summary = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_B2B"], as_index=False)
    .agg(PTS_PER_GAME=("PTS", "mean"), GAMES=("GAME_ID", "count"))
)

# Wide: one row per player — columns 0 and 1 if IS_B2B is int; False/True if bool
wide = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_B2B"])["PTS"]
    .mean()
    .unstack()
    .rename(columns={0: "pts_not_b2b", 1: "pts_b2b"})  # if bool: use False/True as keys
)

# How much better on rest vs B2B (positive = better with rest than on B2B)
wide["not_b2b_minus_b2b"] = wide["pts_not_b2b"] - wide["pts_b2b"]
wide = wide.sort_values("not_b2b_minus_b2b", ascending=False)
wide.sample(10)

,IS_B2B,pts_not_b2b,pts_b2b,not_b2b_minus_b2b
PLAYER_ID,PLAYER_NAME,,,
1627824,Guerschon Yabusele,8.500000,7.380952,1.119048
201976,Patrick Beverley,6.131148,6.416667,-0.285519
1641726,Dereck Lively II,8.500000,8.071429,0.428571
1630591,Jalen Suggs,13.468531,15.416667,-1.948135
1631106,Tari Eason,11.459677,7.133333,4.326344
203957,Dante Exum,7.803030,9.666667,-1.863636
1629627,Zion Williamson,22.750000,20.222222,2.527778
1629684,Grant Williams,9.018018,11.823529,-2.805511
1630573,Sam Hauser,8.907216,9.058824,-0.151607
